%% [markdown]<br>
## --- Generate text with a recurrent neural network (Pytorch) ---<br>
### (Mostly Read & Run)

%%<br>
!wget https://thome.isir.upmc.fr/classes/RITAL/input.txt<br>
!pip install unidecode

%%

In [ ]:
import unidecode
import string
import random
import re
import torch
import torch.nn as nn

Ajout du support GPU (Bonus) pour accÃ©lÃ©rer l'entraÃ®nement

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
all_characters = string.printable
n_characters = len(all_characters)

In [ ]:
file = unidecode.unidecode(open('./input.txt').read()) #clean text => only ascii
file_len = len(file)
print('file_len =', file_len)

%%

In [ ]:
import time, math

et a piece of text

In [ ]:
def random_chunk(chunk_len):
    start_index = random.randint(0, file_len - chunk_len)
    end_index = start_index + chunk_len + 1
    return file[start_index:end_index]

Turn string into list of longs

In [ ]:
def char_tensor(string):
    tensor = torch.zeros(1,len(string)).long()
    for c in range(len(string)):
        tensor[0,c] = all_characters.index(string[c])
    return tensor.to(device) # Transfert automatique sur GPU/CPU

urn a piece of text in train/test

In [ ]:
def random_training_set(chunk_len=200, batch_size=8):
    chunks =[random_chunk(chunk_len) for _ in range(batch_size)]
    inp = torch.cat([char_tensor(chunk[:-1]) for chunk in chunks],dim=0)
    target = torch.cat([char_tensor(chunk[1:]) for chunk in chunks],dim=0)
    return inp, target

In [ ]:
print(random_training_set(10,4)[0].shape)  ## should return [4, 10]

%%

In [ ]:
import torch.nn.functional as F

In [ ]:
class RNN(nn.Module):
    def __init__(self, n_char, hidden_size, output_size, n_layers=1, rnn_cell=nn.RNN):
        """
        Create the network
        """
        super(RNN, self).__init__()
        self.n_char = n_char
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers

        # (batch, chunk_len) -> (batch, chunk_len, hidden_size)
        self.embed = nn.Embedding(num_embeddings=n_char, embedding_dim=hidden_size)

        # (batch, chunk_len, hidden_size)  -> (batch, chunk_len, hidden_size)
        # batch_first=True est crucial ici car nos inputs sont au format (batch, seq, feature)
        self.rnn = rnn_cell(input_size=hidden_size, hidden_size=hidden_size, num_layers=n_layers, batch_first=True)

        #(batch, chunk_len, hidden_size) -> (batch, chunk_len, output_size)
        self.predict = nn.Linear(in_features=hidden_size, out_features=output_size)
    def forward(self, input):
        """
        batched forward: input is (batch > 1,chunk_len)
        """
        input = self.embed(input)
        output, _ = self.rnn(input)
        # torch.tanh est prÃ©fÃ©rÃ© Ã  F.tanh (qui est dÃ©prÃ©ciÃ©)
        output = self.predict(torch.tanh(output))
        return output
    def forward_seq(self, input, hidden=None):
        """
        not batched forward: input is (1,)
        """
        input = self.embed(input)
        # On ajoute la dimension batch et sequence (1, 1, hidden_size)
        output, hidden = self.rnn(input.unsqueeze(0), hidden)
        output = self.predict(torch.tanh(output))
        return output, hidden

%%

In [ ]:
def generate(model, prime_str='A', predict_len=100, temperature=0.8):
    model.eval() # Mode Ã©valuation
    prime_input = char_tensor(prime_str).squeeze(0)
    hidden = None
    predicted = prime_str + ""

    # Use priming string to "build up" hidden state
    for p in range(len(prime_str)-1):
        _, hidden = model.forward_seq(prime_input[p].unsqueeze(0), hidden)
    for p in range(predict_len):
        output, hidden = model.forward_seq(prime_input[-1].unsqueeze(0), hidden)
        # Sample from the network as a multinomial distribution
        output_dist = output.data.view(-1).div(temperature).exp()
        top_i = torch.multinomial(output_dist, 1)[0]
        
        # Add predicted character to string and use as next input
        predicted_char = all_characters[top_i]
        predicted += predicted_char
        prime_input = torch.cat([prime_input, char_tensor(predicted_char).squeeze(0)])
    return predicted

%%

In [ ]:
def time_since(since):
    s = time.time() - since
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

#Parameters

In [ ]:
n_epochs = 20000
print_every = 100
plot_every = 10
hidden_size = 100
n_layers = 2 # DiminuÃ© pour Ã©viter de surcharger (5 Ã©tait Ã©norme pour un simple RNN character-level)
lr = 0.005
batch_size = 16
chunk_len = 80

In [ ]:
model = RNN(n_characters, hidden_size, n_characters, n_layers).to(device) # create model + envoi sur GPU
model_optimizer = torch.optim.Adam(model.parameters(), lr=lr) #create Adam optimizer
criterion = nn.CrossEntropyLoss() #chose criterion

In [ ]:
start = time.time()
all_losses =[]
loss_avg = 0

In [ ]:
def train(inp, target):
    """
    Train sequence for one chunk:
    """
    model.train()
    model_optimizer.zero_grad()
    
    output = model(inp)
    loss = criterion(output.view(batch_size * chunk_len, -1), target.view(-1))
    
    loss.backward()
    model_optimizer.step()
    return loss.item()

In [ ]:
for epoch in range(1, n_epochs + 1):
    loss = train(*random_training_set(chunk_len, batch_size))
    loss_avg += loss
    if epoch % print_every == 0:
        print('[%s (%d %d%%) %.4f]' % (time_since(start), epoch, epoch / n_epochs * 100, loss))
        print(generate(model, 'Wh', 100), '\n')
    if epoch % plot_every == 0:
        all_losses.append(loss_avg / plot_every)
        loss_avg = 0

%%

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
# %matplotlib inline

In [ ]:
plt.figure()
plt.plot(all_losses)
plt.title("Training Loss")
plt.show()

%%

In [ ]:
print(generate(model, 'T', 200, temperature=1))
print("----")
print(generate(model, 'Th', 200, temperature=0.8))
print("----")
print(generate(model, 'Th', 200, temperature=0.5))
print("----")
print(generate(model, 'Th', 200, temperature=0.3))
print("----")
print(generate(model, 'Th', 200, temperature=0.1))